In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

In [2]:
wkend_overall_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkEND-Overall')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkEND-RouteTotal')
df=pd.read_csv("reviewtool_20241223_VTA_ROUTE_DIRECTION_CHECk.csv")

In [3]:
df.drop(columns=['ROUTE_SURVEYEDCode','ROUTE_SURVEYED'],inplace=True)
df.rename(columns={'ROUTE_SURVEYEDCode_New':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_NEW':'ROUTE_SURVEYED'},inplace=True) 

In [4]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:

date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)

In [6]:

def get_day_name(x):
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue

df['Day']=df['Date'].apply(get_day_name)

weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]

weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]

In [7]:
time_column_check=['timeoncode']
time_column=check_all_characters_present(df,time_column_check)

# to get ROUTE_SURVEYEDCode column from database File
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)


In [25]:
weekday_df[time_column[0]].unique()

array(['MID3', 'MID7', 'MID5', 'MID4', nan, 'PM4', 'MID6', 'PM1', 'PM2',
       'PM3', 'PM5', 'MID2', 'PM7', 'AM3', 'PM6', 'PM8', 'MID1', 'PM9',
       'AM2', 'AM1'], dtype=object)

In [8]:
wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)

In [9]:
pre_early_am_values=['AM1'] 
early_am_values=['AM2'] 
am_values=['AM3','AM4','MID1','MID2','MID7'] 
midday_values=['MID3','MID4','MID5','MID6','PM1']
pm_values=['PM2','PM3','PM4','PM5']
evening_values=['PM6','PM7','PM8','PM9']

In [10]:
pre_early_am_column=[0]  #0 is for Pre-Early AM header
early_am_column=[1]  #1 is for Early AM header
am_column=[2] #This is for AM header
midday_colum=[3] #this is for MIDDAY header
pm_column=[4] #this is for PM header
evening_column=[5] #this is for EVENING header

In [11]:
def convert_string_to_integer(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return 0

In [12]:
import math

In [16]:
# Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
new_df=pd.DataFrame()
new_df['ROUTE_SURVEYEDCode']=wkend_overall_df['LS_NAME_CODE']
new_df['CR_PRE_Early_AM']=wkend_overall_df[pre_early_am_column[0]]
new_df['CR_Early_AM']=wkend_overall_df[early_am_column[0]]
new_df['CR_AM_Peak']=wkend_overall_df[am_column[0]]
new_df['CR_Midday']=wkend_overall_df[midday_colum[0]]
new_df['CR_PM_Peak']=wkend_overall_df[pm_column[0]]
new_df['CR_Evening']=wkend_overall_df[evening_column[0]]
# print("new_df_columns",new_df.columns)
new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)

In [17]:
new_df

,ROUTE_SURVEYEDCode,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening
0,VTA_2_20_00,0.000000,0.315153,0.415429,0.329478,0.114601,0.000000
1,VTA_2_20_01,0.000000,0.530030,0.329478,0.272178,0.143251,0.000000
2,VTA_2_21_00,0.014325,3.251807,5.615456,5.042450,1.260613,0.000000
3,VTA_2_21_01,0.157577,3.667236,5.386254,4.383494,1.074386,0.000000
4,VTA_2_22_00,1.017085,14.840848,23.736761,23.120780,14.941124,2.048495
...,...,...,...,...,...,...,...
99,VTA_2_BlueLine_01,0.000000,5.500855,7.477724,9.683796,7.176896,0.257853
100,VTA_2_GreenLine_00,0.085951,5.085426,7.449074,7.449074,4.698647,0.214877
101,VTA_2_GreenLine_01,0.257853,3.194507,6.990670,7.348798,4.311868,0.028650
102,VTA_2_OrangeLine_00,0.214877,6.503615,13.150481,14.625971,6.646866,0.000000


In [21]:
new_df_1

,ROUTE_SURVEYEDCode,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening
0,VTA_2_20_00,0.0,1.0,1.0,1.0,1.0,0.0
1,VTA_2_20_01,0.0,1.0,1.0,1.0,1.0,0.0
2,VTA_2_21_00,1.0,4.0,6.0,6.0,2.0,0.0
3,VTA_2_21_01,1.0,4.0,6.0,5.0,2.0,0.0
4,VTA_2_22_00,2.0,15.0,24.0,24.0,15.0,3.0
...,...,...,...,...,...,...,...
99,VTA_2_BlueLine_01,0.0,6.0,8.0,10.0,8.0,1.0
100,VTA_2_GreenLine_00,1.0,6.0,8.0,8.0,5.0,1.0
101,VTA_2_GreenLine_01,1.0,4.0,7.0,8.0,5.0,1.0
102,VTA_2_OrangeLine_00,1.0,7.0,14.0,15.0,7.0,0.0


In [20]:
# Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
new_df_1=pd.DataFrame()
new_df_1['ROUTE_SURVEYEDCode']=wkend_overall_df['LS_NAME_CODE']
new_df_1['CR_PRE_Early_AM']=wkend_overall_df[pre_early_am_column[0]].apply(math.ceil)
new_df_1['CR_Early_AM']=wkend_overall_df[early_am_column[0]].apply(math.ceil)
new_df_1['CR_AM_Peak']=wkend_overall_df[am_column[0]].apply(math.ceil)
new_df_1['CR_Midday']=wkend_overall_df[midday_colum[0]].apply(math.ceil)
new_df_1['CR_PM_Peak']=wkend_overall_df[pm_column[0]].apply(math.ceil)
new_df_1['CR_Evening']=wkend_overall_df[evening_column[0]].apply(math.ceil)
# print("new_df_columns",new_df.columns)
new_df_1[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df_1[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)

In [14]:
new_df.fillna(0,inplace=True)

In [15]:
def get_counts_and_ids(time_values):
    # Just for SALEM
    # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
    subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
    subset_df=subset_df.drop_duplicates(subset='id')
    count = subset_df.shape[0]
    ids = subset_df['id'].values
    return count, ids

In [16]:
for index, row in new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
    early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    
    new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
    # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
    new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
    new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
    
#     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
    route_code_level_df=pd.DataFrame()

    unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

    route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

    for index, row in new_df.iterrows():
        pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
#         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
#         new_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))
    

#     weekend_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)
#     route_code_level_df=pd.merge(route_code_level_df,weekend_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']],on='ROUTE_SURVEYEDCode')
#     weekend_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)

#     for index , row in route_code_level_df.iterrows():
#         subset_code_df=new_df[new_df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode']]
#         # sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()



#         sum_per_route_code_cr = subset_code_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
#         # sum_per_route_cr = subset_df[['CR_EARLY_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
#         sum_per_route_code_db = subset_code_df[['DB_PRE_Early_AM_Peak','DB_Early_AM_Peak','DB_AM_Peak',
#                                            'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()

#         route_code_level_df.loc[index,'CR_PRE_Early_AM']=sum_per_route_code_cr['CR_PRE_Early_AM']
#         route_code_level_df.loc[index,'CR_Early_AM']=sum_per_route_code_cr['CR_Early_AM']
#         route_code_level_df.loc[index,'CR_AM_Peak']=sum_per_route_code_cr['CR_AM_Peak']
#         route_code_level_df.loc[index,'CR_Midday']=sum_per_route_code_cr['CR_Midday']
#         route_code_level_df.loc[index,'CR_PM_Peak']=sum_per_route_code_cr['CR_PM_Peak']
#         route_code_level_df.loc[index,'CR_Evening']=sum_per_route_code_cr['CR_Evening']
#         route_code_level_df.loc[index,'CR_Total']=sum_per_route_code_cr['CR_Total']
#         # route_level_df.loc[index,'CR_Overall_Goal']=sum_per_route_cr['Overall Goal']

#         route_code_level_df.loc[index,'DB_PRE_Early_AM_Peak']=sum_per_route_code_db['DB_PRE_Early_AM_Peak']
#         route_code_level_df.loc[index,'DB_Early_AM_Peak']=sum_per_route_code_db['DB_Early_AM_Peak']
#         route_code_level_df.loc[index,'DB_AM_Peak']=sum_per_route_code_db['DB_AM_Peak']
#         route_code_level_df.loc[index,'DB_Midday']=sum_per_route_code_db['DB_Midday']
#         route_code_level_df.loc[index,'DB_PM_Peak']=sum_per_route_code_db['DB_PM_Peak']
#         route_code_level_df.loc[index,'DB_Evening']=sum_per_route_code_db['DB_Evening']
#         route_code_level_df.loc[index,'DB_Total']=sum_per_route_code_db['DB_Total']   

#     # route_level_df.to_csv('Route Level Comparison(Value_Check).csv',index=False)

#     # calculating the difference between values of database and compeletion report for Route_Level
#     for index, row in route_code_level_df.iterrows():
#         pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
#         early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
#         am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
#         midday_diff=row['CR_Midday']-row['DB_Midday']    
#         pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
#         evening_diff=row['CR_Evening']-row['DB_Evening']
#         total_diff=row['CR_Total']-row['DB_Total']
#         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
#         route_code_level_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
#         route_code_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
#         route_code_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
#         route_code_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
#         route_code_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
#         route_code_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
#         # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
#         route_code_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
#         route_code_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))

#     #     return route_code_level_df


In [17]:
route_code_level_df

,ROUTE_SURVEYEDCode
0,VTA_2_20_00
1,VTA_2_20_01
2,VTA_2_21_00
3,VTA_2_21_01
4,VTA_2_22_00
...,...
99,VTA_2_BlueLine_01
100,VTA_2_GreenLine_00
101,VTA_2_GreenLine_01
102,VTA_2_OrangeLine_00


In [18]:
new_df.to_csv('Route_Code_Level_Comparison.csv',index=False)
new_df

,ROUTE_SURVEYEDCode,CR_PRE_Early_AM,CR_Early_AM,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_PRE_Early_AM_Peak,DB_Early_AM_Peak,...,DB_PM_Peak,DB_Evening,DB_Total,PRE_Early_AM_DIFFERENCE,Early_AM_DIFFERENCE,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,VTA_2_20_00,0.000000,0.315153,0.415429,0.329478,0.114601,0.000000,1.174662,1.0,0.0,...,8.0,0.0,27.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,VTA_2_20_01,0.000000,0.530030,0.329478,0.272178,0.143251,0.000000,1.274938,0.0,0.0,...,5.0,0.0,37.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,VTA_2_21_00,0.014325,3.251807,5.615456,5.042450,1.260613,0.000000,15.184651,0.0,0.0,...,17.0,10.0,93.0,1.0,4.0,0.0,0.0,0.0,0.0,5.0
3,VTA_2_21_01,0.157577,3.667236,5.386254,4.383494,1.074386,0.000000,14.668946,1.0,0.0,...,16.0,6.0,61.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0
4,VTA_2_22_00,1.017085,14.840848,23.736761,23.120780,14.941124,2.048495,79.705093,0.0,0.0,...,108.0,46.0,339.0,2.0,15.0,0.0,0.0,0.0,0.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,VTA_2_BlueLine_01,0.000000,5.500855,7.477724,9.683796,7.176896,0.257853,30.097124,2.0,1.0,...,64.0,45.0,278.0,0.0,5.0,0.0,0.0,0.0,0.0,5.0
100,VTA_2_GreenLine_00,0.085951,5.085426,7.449074,7.449074,4.698647,0.214877,24.983049,0.0,2.0,...,55.0,21.0,189.0,1.0,4.0,0.0,0.0,0.0,0.0,5.0
101,VTA_2_GreenLine_01,0.257853,3.194507,6.990670,7.348798,4.311868,0.028650,22.132345,0.0,1.0,...,54.0,22.0,178.0,1.0,3.0,0.0,0.0,0.0,0.0,4.0
102,VTA_2_OrangeLine_00,0.214877,6.503615,13.150481,14.625971,6.646866,0.000000,41.141809,1.0,0.0,...,89.0,33.0,229.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0


In [19]:
new_df.columns

Index(['ROUTE_SURVEYEDCode', 'CR_PRE_Early_AM', 'CR_Early_AM', 'CR_AM_Peak',
       'CR_Midday', 'CR_PM_Peak', 'CR_Evening', 'CR_Total',
       'DB_PRE_Early_AM_Peak', 'DB_Early_AM_Peak', 'DB_AM_Peak', 'DB_Midday',
       'DB_PM_Peak', 'DB_Evening', 'DB_Total', 'PRE_Early_AM_DIFFERENCE',
       'Early_AM_DIFFERENCE', 'AM_DIFFERENCE', 'Midday_DIFFERENCE',
       'PM_DIFFERENCE', 'Evening_DIFFERENCE', 'Total_DIFFERENCE'],
      dtype='object')